In [1]:
import CalculatedFieldSubroutines as cfs

#

import numpy as np

import pandas as pd

#

import os

In [2]:
gmIDs = cfs.list_whitelisted_gmIDs()

topics = cfs.list_topics()

print( topics )

['/apollo/canbus/chassis', '/apollo/drive/event', '/apollo/sensor/gnss/best/pose', '/apollo/perception/traffic/light']


In [3]:
def CreatePreprocessedMovingDataFolder( moving_window, expansion_window = 1 ): # sec

    expansion_window_ns = expansion_window * 1e9

    moving_window_ns = moving_window * 1e9

    #

    for index, gmID in enumerate( gmIDs ):

        chassis_df = cfs.retrieve_gmID_topic( gmID, '/apollo/canbus/chassis' )

        pose_df = cfs.retrieve_gmID_topic( gmID, '/apollo/sensor/gnss/best/pose' )

        #

        chassis_df = chassis_df.sort_values( 'time' )

        pose_df = pose_df.sort_values( 'time' )

        #

        cfs.Index( chassis_df )

        #

        cfs.NormalizedTime( chassis_df )

        #

        cfs.BinaryDrivingMode( chassis_df )

        cfs.TernaryDrivingModeTransition( chassis_df )

        cfs.BinaryDisengagement( chassis_df )

        cfs.BinaryDisengagementExpanded( chassis_df, moving_colname = 'time', window = expansion_window_ns )

        cfs.DisengagementID( chassis_df, expanded = False )

        cfs.DisengagementID( chassis_df, expanded = True )

        #

        cfs.Acceleration( chassis_df )

        #

        chassis_df = chassis_df.drop( [ 'drivingMode', 'TernaryDrivingModeTransition', 'signal.turnSignal' ], axis = 1 )

        #

        cfs.LatLonTotalStdDev( pose_df )

        #

        cfs.ProgressAlongRoute_v2( pose_df )

        #

        pose_df = pose_df.drop( [ 'heightMsl', 'groupMetadataID', 'latitudeStdDev', 'heightStdDev', 'longitudeStdDev', \
                                  'PartitionNumber', 'numSatsInSolution' ], axis = 1 )

        #

        cfs.ChassisBestPoseMatchedTime( chassis_df, pose_df )

        merged_df = pd.merge( chassis_df, pose_df, on = 'ChassisBestPoseMatchedTime', how = 'inner' )

        merged_df = merged_df.rename( columns = { 'time_x' : 'time' } )

        merged_df = merged_df.drop( [ 'ChassisBestPoseMatchedTime', 'time_y' ], axis = 1 )

        #

        merged_df = merged_df[ [ 'Ind', 'groupMetadataID', 'time', 'NormalizedTime', 'BinaryDrivingMode', 'BinaryDisengagement', \
                                 'BinaryDisengagementExpanded', 'DisengagementID', 'DisengagementExpandedID', 'latitude', \
                                 'longitude', 'ProgressAlongRoute', 'solStatus', 'extendedSolutionStatus', \
                                 'solType', 'speedMps', 'Acceleration', 'brakePercentage', 'throttlePercentage', \
                                 'steeringPercentage', 'LatLonTotalStdDev' ] ]

        #

        conditional_string = ''

        if ( moving_window > 0 ):

            function_desired_colnames = { cfs.majority_value : [ 'solStatus', 'extendedSolutionStatus', 'solType' ], 
                                          np.mean : [ 'speedMps', 'Acceleration', 'brakePercentage', 'throttlePercentage', \
                                                      'steeringPercentage', 'LatLonTotalStdDev' ] }

            function_suffixes = { cfs.majority_value : 'majority', 
                                  np.mean : 'mean' }

            #

            conditional_string = '_majority'

            #

            cfs.GeneralizedMovingFunction( df = merged_df, \
                                           moving_colname = 'time', \
                                           relative_moving_window_interval = ( -1 * moving_window_ns, 0 ), \
                                           function_desired_colnames = function_desired_colnames, \
                                           function_suffixes = function_suffixes )

            #

            merged_df = merged_df.drop( [ 'speedMps', 'Acceleration', 'brakePercentage', 'throttlePercentage', \
                                          'steeringPercentage', 'LatLonTotalStdDev', 'solStatus', \
                                          'extendedSolutionStatus', 'solType' ], axis = 1 )

            #

        cfs.OneHotEncoder( df = merged_df, 
                           colname_to_encode = f'solStatus{ conditional_string }',
                           unique_values = [ 'COV_TRACE', 'INSUFFICIENT_OBS', 'INTEGRITY_WARNING', 'SOL_COMPUTED' ] )

        cfs.OneHotEncoder( df = merged_df, 
                           colname_to_encode = f'extendedSolutionStatus{ conditional_string }',
                           unique_values = [ 0, 2, 6, 8, 32, 33, 130, 134, 136 ] )

        cfs.OneHotEncoder( df = merged_df, 
                           colname_to_encode = f'solType{ conditional_string }',
                           unique_values = [ 'L1_FLOAT', 'NARROW_FLOAT', 'NARROW_INT', 'PSRDIFF', 'SINGLE', 'WIDE_INT' ] )

        #

        filepath = f'{ cfs.origin_dir() }/Preprocessed_Moving_Data_v3/{ moving_window }sec_moving_window/{ gmID }'

        os.makedirs( filepath, exist_ok = True )

        #

        merged_df.to_csv( f'{ filepath }/{ gmID }.csv', index = False )

In [4]:
for sec in [ 2, 4, 0 ]:

    CreatePreprocessedMovingDataFolder( moving_window = sec )

    print( f'{ sec } second moving window data done' )

2 second moving window data done
4 second moving window data done
0 second moving window data done
